# 01 | Build the USD price panel

Build a panel of traded closes for ten major banks from the final session of 2005 through 2008. Historical identifiers pin the securities through ticker changes and delistings. The unit of analysis is one NYSE close-to-close interval.

In [1]:
from pathlib import Path
from contextlib import closing
from datetime import datetime, timezone
from getpass import getpass
from io import BytesIO
import hashlib
import json
import os

import exchange_calendars as xcals
import numpy as np
import pandas as pd
import psycopg2
import requests

cache = Path("data/private")
cache.mkdir(parents=True, exist_ok=True)
results = Path("results")
results.mkdir(exist_ok=True)

start, end = "2005-12-30", "2008-12-31"
nyse = xcals.get_calendar("XNYS", start=start, end=end)
sessions = nyse.sessions_in_range(start, end).rename("date")

## Historical security map

The US securities are five common stocks, two NYSE-listed foreign ordinary shares (UBS AG and Deutsche Bank), and two ADSs (Barclays and Credit Suisse). BNP Paribas is the Paris-listed ordinary share. The historical Compustat North America ADR query is included below so the choice of listing can be audited.

In [2]:
security_map = pd.DataFrame(
    [
        ("GS", "Goldman Sachs", 86868, "GS", "GOLDMAN SACHS"),
        ("UBS", "UBS", 88239, "UBS", "U B S AG"),
        ("JPM", "JPMorgan Chase", 47896, "JPM", "JPMORGAN CHASE"),
        ("C", "Citigroup", 70519, "C", "CITIGROUP"),
        ("BCS", "Barclays", 69761, "BCS", "BARCLAYS"),
        ("MS", "Morgan Stanley", 69032, "MWD / MS", "MORGAN STANLEY"),
        ("DB", "Deutsche Bank", 89199, "DB", "DEUTSCHE BANK"),
        ("BAC", "Bank of America", 59408, "BAC", "BANK OF AMERICA"),
        ("BNP", "BNP Paribas", None, "BNP", "BNP PARIBAS"),
        ("CS", "Credit Suisse", 89154, "CSR / CS", "CREDIT SUISSE"),
    ],
    columns=["bank_code", "bank", "permno", "period_ticker", "issuer_check"],
).set_index("bank_code")
us_securities = security_map.drop(index="BNP")
permnos = tuple(us_securities["permno"].astype(int))

## Download inputs and cache the responses

A first run uses WRDS PostgreSQL and the public ECB API. WRDS requires access to CRSP US Stock, Compustat Global Security Daily and Compustat North America Security Daily. The database client reads a local `.pgpass` or `PGPASSFILE`; the notebook never records a password. Subsequent runs reuse the CSVs. Set `BANK_VAR_REFRESH=1` to fetch the inputs again.

In [3]:
wrds_queries = {
    "us_closes": ("""
        SELECT permno, date, prc, vol, cfacshr, cfacpr
        FROM crsp.dsf
        WHERE permno IN %s AND date BETWEEN %s AND %s
        ORDER BY permno, date
    """, (permnos, start, end)),
    "us_identity": ("""
        SELECT permno, ticker, comnam, exchcd, shrcd, ncusip, namedt, nameendt
        FROM crsp.dsenames
        WHERE permno IN %s AND nameendt >= %s AND namedt <= %s
        ORDER BY permno, namedt
    """, (permnos, start, end)),
    "us_actions": ("""
        SELECT permno, exdt, distcd, facpr, facshr
        FROM crsp.dsedist
        WHERE permno IN %s AND exdt BETWEEN %s AND %s
              AND (facpr <> 0 OR facshr <> 0)
        ORDER BY permno, exdt, distcd
    """, (permnos, start, end)),
    "paris_closes": ("""
        SELECT datadate AS date, curcdd AS currency, isin, exchg, prccd, cshtrd,
               prcstd, ajexdi, split, splitf
        FROM comp.g_secd
        WHERE gvkey = '015532' AND iid = '04W' AND datadate BETWEEN %s AND %s
        ORDER BY datadate
    """, (start, end)),
    "bnp_adr_screen": ("""
        SELECT tic, iid, curcdd, exchg, prcstd, count(*) AS rows,
               count(prccd) AS priced_rows,
               count(*) FILTER (WHERE cshtrd > 0) AS positive_volume_rows
        FROM comp.secd
        WHERE gvkey = '015532' AND iid IN ('01', '91') AND datadate BETWEEN %s AND %s
        GROUP BY tic, iid, curcdd, exchg, prcstd
        ORDER BY iid, prcstd
    """, (start, end)),
}

def save_cache(path, content):
    temporary = path.with_suffix(".tmp")
    temporary.write_bytes(content)
    temporary.replace(path)

refresh = os.environ.get("BANK_VAR_REFRESH") == "1"
missing_wrds = [name for name in wrds_queries if refresh or not (cache / f"{name}.csv").is_file()]
if missing_wrds:
    wrds_user = os.environ.get("WRDS_USERNAME") or input("WRDS username: ").strip()
    connection_options = dict(
        host="wrds-pgdata.wharton.upenn.edu", port=9737, dbname="wrds",
        user=wrds_user, sslmode="require", connect_timeout=20,
    )
    try:
        connection = psycopg2.connect(**connection_options)
    except psycopg2.OperationalError as error:
        if "no password supplied" not in str(error):
            raise
        connection = psycopg2.connect(**connection_options, password=getpass("WRDS password: "))
    with closing(connection), connection.cursor() as cursor:
        cursor.execute("SET statement_timeout = '120s'")
        downloads = {}
        for name in missing_wrds:
            sql, parameters = wrds_queries[name]
            cursor.execute(sql, parameters)
            columns = [column.name for column in cursor.description]
            frame = pd.DataFrame(cursor.fetchall(), columns=columns)
            if frame.empty:
                raise ValueError(f"WRDS returned no rows for {name}.")
            downloads[name] = frame.to_csv(index=False).encode()
        for name, content in downloads.items():
            save_cache(cache / f"{name}.csv", content)

fx_url = "https://data-api.ecb.europa.eu/service/data/EXR/D.USD.EUR.SP00.A"
fx_file = cache / "ecb_usd_per_eur.csv"
if refresh or not fx_file.is_file():
    response = requests.get(
        fx_url, params={"startPeriod": start, "endPeriod": end, "format": "csvdata"}, timeout=45
    )
    response.raise_for_status()
    assert {"TIME_PERIOD", "OBS_VALUE"} <= set(pd.read_csv(BytesIO(response.content), nrows=1))
    save_cache(fx_file, response.content)

input_files = [f"{name}.csv" for name in wrds_queries] + [fx_file.name]
if missing_wrds or refresh or not (cache / "input_manifest.json").is_file():
    manifest = {
        "cached_utc": datetime.now(timezone.utc).isoformat(),
        "sha256": {name: hashlib.sha256((cache / name).read_bytes()).hexdigest() for name in input_files},
    }
    save_cache(cache / "input_manifest.json", json.dumps(manifest, indent=2).encode())

## Verify securities and build USD marks

The CRSP screen requires a positive traded close. Its cumulative share factor (`CFACSHR`) adjusts for splits and stock dividends. It excludes non-share price factors such as the Morgan Stanley Discover spin-off or UBS subscription rights. Divide the close by the cumulative share factor normalized to the first date.

Use BNP's Paris close and the ECB USD-per-EUR rate from the same calendar date. The Paris close (normally 17:30 local) and the ECB reference fixing (14:15 CET in the historical series metadata) are both available by the scheduled NYSE close. The ADR had no positive volume in the candidate history. BNP's only `AJEXDI` move reflects its March 2006 rights offering, so we use the raw close for the split-only convention. The underlying shares have no split in the window.

In [4]:
identities = pd.read_csv(cache / "us_identity.csv")
for code, security in us_securities.iterrows():
    history = identities.loc[identities["permno"].eq(security["permno"])]
    assert set(history["ticker"]) == set(security["period_ticker"].split(" / ")), code
    assert history["comnam"].str.startswith(security["issuer_check"]).all(), code
    assert history["exchcd"].eq(1).all(), code
    assert history["shrcd"].eq({"UBS": 12, "DB": 12, "BCS": 31, "CS": 31}.get(code, 11)).all()
assert identities["permno"].nunique() == 9

us = pd.read_csv(cache / "us_closes.csv", parse_dates=["date"])
assert not us.duplicated(["date", "permno"]).any()
assert pd.DatetimeIndex(us["date"].unique()).difference(sessions).empty
assert us["cfacshr"].gt(0).all()
share_factor_at_start = us.groupby("permno")["cfacshr"].transform("first")
us["usd_close"] = us["prc"].where(us["prc"].gt(0) & us["vol"].gt(0))
us["usd_close"] = us["usd_close"] / (us["cfacshr"] / share_factor_at_start)
prices = us.pivot(index="date", columns="permno", values="usd_close")
prices = prices.rename(columns=dict(zip(permnos, us_securities.index))).reindex(sessions)

adr_screen = pd.read_csv(cache / "bnp_adr_screen.csv", dtype={"iid": str})
assert adr_screen.loc[adr_screen["iid"].eq("91"), "positive_volume_rows"].sum() == 0
paris = pd.read_csv(cache / "paris_closes.csv", parse_dates=["date"]).set_index("date")
assert paris.index.is_unique and paris.index.is_monotonic_increasing
assert paris["isin"].eq("FR0000131104").all() and paris["exchg"].eq(286).all()
assert paris["currency"].eq("EUR").all()
paris_actions = paris.loc[paris["split"].notna(), ["split", "splitf"]]
assert paris_actions.index.tolist() == [pd.Timestamp("2006-03-07")]
assert paris_actions["splitf"].tolist() == ["OR"]
factor_changes = paris["ajexdi"].ne(paris["ajexdi"].shift()).iloc[1:]
assert factor_changes[factor_changes].index.equals(paris_actions.index)

fx = pd.read_csv(fx_file, parse_dates=["TIME_PERIOD"]).set_index("TIME_PERIOD")
assert fx.index.is_unique and fx["CURRENCY"].eq("USD").all() and fx["CURRENCY_DENOM"].eq("EUR").all()
assert fx["OBS_VALUE"].dropna().gt(0).all()
paris_close = paris["prccd"].where(paris["prccd"].gt(0) & paris["cshtrd"].gt(0))
prices["BNP"] = paris_close.reindex(sessions) * fx["OBS_VALUE"].reindex(sessions)
prices = prices.reindex(columns=security_map.index).rename_axis(columns=None)
assert prices.shape[1] == 10 and prices.notna().any().all()
assert np.isfinite(prices.stack()).all() and prices.stack().gt(0).all()

display(adr_screen)
display(identities.groupby("permno").agg(
    tickers=("ticker", lambda tickers: " / ".join(dict.fromkeys(tickers))),
    name=("comnam", "last"), exchange_code=("exchcd", "first"), share_code=("shrcd", "first"),
))

,tic,iid,curcdd,exchg,prcstd,rows,priced_rows,positive_volume_rows
0,BNPQF,01,USD,19,3,206,206,206
1,BNPQY,91,USD,19,4,756,756,0


,tickers,name,exchange_code,share_code
permno,,,,
47896,JPM,JPMORGAN CHASE & CO,1,11
59408,BAC,BANK OF AMERICA CORP,1,11
69032,MWD / MS,MORGAN STANLEY DEAN WITTER & CO,1,11
69761,BCS,BARCLAYS PLC,1,31
70519,C,CITIGROUP INC,1,11
86868,GS,GOLDMAN SACHS GROUP INC,1,11
88239,UBS,U B S AG,1,12
89154,CSR / CS,CREDIT SUISSE GROUP,1,31
89199,DB,DEUTSCHE BANK A G,1,12


## Coverage and jump audit

A missing quote stays missing. A one-day return requires prices on consecutive NYSE sessions. Full portfolio returns require all ten pairs. Review the largest moves alongside all relevant capital actions. The portfolio uses the parent-stock marks through rights and spin-offs and holds no distributed securities.

In [5]:
coverage = security_map[["bank", "period_ticker"]].copy()
coverage["security_id"] = us_securities["permno"].astype(int).map(lambda x: f"PERMNO {x}")
coverage.loc["BNP", "security_id"] = "FR0000131104 / 015532-04W"
coverage["venue"] = "NYSE"
coverage.loc["BNP", "venue"] = "Euronext Paris"
coverage["source"] = "CRSP via WRDS"
coverage.loc["BNP", "source"] = "Compustat Global via WRDS + ECB"
coverage["native_currency"] = "USD"
coverage.loc["BNP", "native_currency"] = "EUR"
coverage["price_field"] = "PRC / normalized CFACSHR"
coverage.loc["BNP", "price_field"] = "PRCCD × ECB USD/EUR"
coverage["first_valid_mark"] = prices.apply(pd.Series.first_valid_index)
coverage["last_valid_mark"] = prices.apply(pd.Series.last_valid_index)
coverage["valid_marks"] = prices.count()
coverage["missing_marks"] = prices.isna().sum()
coverage.to_csv(results / "data_coverage.csv", date_format="%Y-%m-%d")
display(coverage)

stock_returns = prices.pct_change(fill_method=None)
large_moves = stock_returns.stack().rename("return").rename_axis(["date", "bank_code"]).reset_index()
large_moves = large_moves.loc[large_moves["return"].abs().ge(0.15)]
large_moves = large_moves.sort_values("return", key=abs, ascending=False)
actions = pd.read_csv(cache / "us_actions.csv", parse_dates=["exdt"])
actions["bank_code"] = actions["permno"].map(dict(zip(permnos, us_securities.index)))
factor_moves = us["cfacshr"].div(us.groupby("permno")["cfacshr"].shift()).dropna()
factor_moves = factor_moves.loc[~np.isclose(factor_moves, 1)]
split_actions = actions.loc[actions["facshr"].ne(0)]
assert set(zip(us.loc[factor_moves.index, "permno"], us.loc[factor_moves.index, "date"])) == set(
    zip(split_actions["permno"], split_actions["exdt"])
)
display(actions[["bank_code", "exdt", "distcd", "facpr", "facshr"]])
display(paris_actions)

quality = pd.DataFrame(
    {
        "nyse_sessions": prices.groupby(prices.index.year).size(),
        "complete_marks": prices.notna().all(axis=1).groupby(prices.index.year).sum(),
        "complete_one_day_returns": stock_returns.notna().all(axis=1).groupby(prices.index.year).sum(),
    }
).rename_axis("year")
gap = prices.index[prices.isna().any(axis=1)]
display(quality)
print("Missing NYSE-session marks:", ", ".join(gap.strftime("%Y-%m-%d")) or "None")
print(f"{len(large_moves)} bank moves of at least 15% in absolute value. Review in the private jump audit.")

prices.to_csv(cache / "usd_prices.csv")
large_moves.to_csv(cache / "jump_audit.csv", index=False)
quality.to_csv(results / "session_counts.csv")

,bank,period_ticker,security_id,venue,source,native_currency,price_field,first_valid_mark,last_valid_mark,valid_marks,missing_marks
bank_code,,,,,,,,,,,
GS,Goldman Sachs,GS,PERMNO 86868,NYSE,CRSP via WRDS,USD,PRC / normalized CFACSHR,2005-12-30,2008-12-31,756,0
UBS,UBS,UBS,PERMNO 88239,NYSE,CRSP via WRDS,USD,PRC / normalized CFACSHR,2005-12-30,2008-12-31,756,0
JPM,JPMorgan Chase,JPM,PERMNO 47896,NYSE,CRSP via WRDS,USD,PRC / normalized CFACSHR,2005-12-30,2008-12-31,756,0
C,Citigroup,C,PERMNO 70519,NYSE,CRSP via WRDS,USD,PRC / normalized CFACSHR,2005-12-30,2008-12-31,756,0
BCS,Barclays,BCS,PERMNO 69761,NYSE,CRSP via WRDS,USD,PRC / normalized CFACSHR,2005-12-30,2008-12-31,756,0
MS,Morgan Stanley,MWD / MS,PERMNO 69032,NYSE,CRSP via WRDS,USD,PRC / normalized CFACSHR,2005-12-30,2008-12-31,756,0
DB,Deutsche Bank,DB,PERMNO 89199,NYSE,CRSP via WRDS,USD,PRC / normalized CFACSHR,2005-12-30,2008-12-31,756,0
BAC,Bank of America,BAC,PERMNO 59408,NYSE,CRSP via WRDS,USD,PRC / normalized CFACSHR,2005-12-30,2008-12-31,756,0
BNP,BNP Paribas,BNP,FR0000131104 / 015532-04W,Euronext Paris,Compustat Global via WRDS + ECB,EUR,PRCCD × ECB USD/EUR,2005-12-30,2008-12-31,747,9


,bank_code,exdt,distcd,facpr,facshr
0,MS,2007-07-02,3763,0.19277,0.00
1,UBS,2006-07-10,5523,1.00000,1.00
2,UBS,2008-04-28,5523,0.05000,0.05
3,UBS,2008-05-27,4533,0.06696,0.00


,split,splitf
date,,
2006-03-07,1.01106,OR


,nyse_sessions,complete_marks,complete_one_day_returns
year,,,
2005,1,1,0
2006,251,248,245
2007,251,248,245
2008,253,250,247


Missing NYSE-session marks: 2006-04-17, 2006-05-01, 2006-12-26, 2007-04-09, 2007-05-01, 2007-12-26, 2008-03-24, 2008-05-01, 2008-12-26
82 bank moves of at least 15% in absolute value. Review in the private jump audit.
